<h2>Engineer advanced lag-based features and rolling window aggregations for time-series forecasting, followed by testing stationarity and applying time-based cross-validation to ensure robustness.</h2>

In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)
dates = pd.date_range(start='2026-01-01', periods=100, freq='D')
trend = np.linspace(10, 50, 100)
seasonality = 10 * np.sin(np.linspace(0, 20, 100))
noise = np.random.normal(0, 2, 100)

In [3]:
df = pd.DataFrame({
    'date': dates,
    'revenue': trend + seasonality + noise
}).set_index('date')

In [5]:
print("Time Series Sample:")
print(df.head())

Time Series Sample:
              revenue
date                 
2026-01-01  10.993428
2026-01-02  12.134000
2026-01-03  16.034824
2026-01-04  19.954522
2026-01-05  18.377480


In [7]:
df['lag_1'] = df['revenue'].shift(1)

In [8]:
df['lag_7'] = df['revenue'].shift(7)

In [9]:
df['lag_14'] = df['revenue'].shift(14)

In [10]:
df['rolling_mean_7'] = df['revenue'].shift(1).rolling(window=7).mean()

In [11]:
df['rolling_std_7'] = df['revenue'].shift(1).rolling(window=7).std()

In [12]:
df['rolling_mean_14'] = df['revenue'].shift(1).rolling(window=14).mean()

In [13]:
df_clean = df.dropna()

In [14]:
print("Engineered Features Sample:")
print(df_clean.head())

Engineered Features Sample:
              revenue      lag_1      lag_7     lag_14  rolling_mean_7  \
date                                                                     
2026-01-15  15.288820  16.354185  24.240699  10.993428       21.826733   
2026-01-16  16.046631  15.288820  22.283086  12.134000       20.547894   
2026-01-17  13.532923  16.046631  24.417043  16.034824       19.656971   
2026-01-18  14.611311  13.532923  22.120623  19.954522       18.102097   
2026-01-19  10.708378  14.611311  21.465186  18.377480       17.029338   

            rolling_std_7  rolling_mean_14  
date                                        
2026-01-15       2.674491        19.660605  
2026-01-16       3.376054        19.967419  
2026-01-17       3.653326        20.246892  
2026-01-18       3.605620        20.068185  
2026-01-19       3.316231        19.686527  


In [15]:
from statsmodels.tsa.stattools import adfuller

In [16]:
adf_result = adfuller(df_clean['revenue'])

In [17]:
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.4f}")

ADF Statistic: -1.1246
p-value: 0.7052


In [18]:
if adf_result[1] <= 0.05:
    print("Result: Series is Stationary (p <= 0.05)")
else:
    print("Result: Series is Non-Stationary (p > 0.05). Differencing required.")

Result: Series is Non-Stationary (p > 0.05). Differencing required.


In [19]:
from sklearn.model_selection import TimeSeriesSplit

In [20]:
X = df_clean.drop(columns=['revenue'])
y = df_clean['revenue']

In [21]:
tscv = TimeSeriesSplit(n_splits=5)

In [22]:
for fold, (train_index, test_index) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

In [23]:
print(f"Fold {fold}: Train Range ({X_train.index.min().strftime('%Y-%m-%d')} to {X_train.index.max().strftime('%Y-%m-%d')}) | "
          f"Test Range ({X_test.index.min().strftime('%Y-%m-%d')} to {X_test.index.max().strftime('%Y-%m-%d')})")

Fold 5: Train Range (2026-01-15 to 2026-03-27) | Test Range (2026-03-28 to 2026-04-10)
